# ZuCo text-only sentiment baseline

Frozen and fine-tuned BERT baselines for ZuCo Task 1 sentiment, run with
stratified k-fold cross-validation.

Use a **GPU runtime**: *Runtime -> Change runtime type -> GPU*.

## 1. Get the code

Clones the repo on first run and pulls the latest commit on every run after,
so updates pushed to GitHub show up here by re-running this cell.

In [ ]:
REPO = "https://github.com/parmisbathaeiyan/zuco-text-baseline.git"
NAME = "zuco-text-baseline"

import os
if not os.path.exists(NAME):
    !git clone -q $REPO
else:
    !git -C $NAME pull -q
%cd $NAME

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Where to save results

Results (JSON summaries + confusion-matrix PNGs) are written to `OUTPUT_DIR`.

Mounting Drive is optional but recommended: the local Colab disk is wiped when
the runtime disconnects, whereas a Drive folder keeps your results. To stay on
local disk instead, just set `OUTPUT_DIR = "results"` and skip the mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/zuco_results'
# OUTPUT_DIR = 'results'  # uncomment to keep results on the local Colab disk

## 4. Frozen baseline

Encoder weights stay fixed; only a linear probe on top of the pooled
features is trained.

In [ ]:
!python run.py --mode frozen --model-name bert-base-uncased --output-dir "$OUTPUT_DIR"

## 5. Fine-tuned baseline

The whole encoder is fine-tuned. This is the real text-only ceiling.

In [ ]:
!python run.py --mode finetune --model-name bert-base-uncased --output-dir "$OUTPUT_DIR"

## 6. (Optional) sweep a few backbones

RoBERTa often edges out BERT on sentiment; DistilBERT is the quick option.

In [ ]:
!python run.py --mode both --output-dir "$OUTPUT_DIR" \
    --model-name bert-base-uncased roberta-base distilbert-base-uncased

## 7. Inspect results

Prints the averaged scores from every saved run and shows the confusion
matrices.

In [ ]:
import json, glob
for path in sorted(glob.glob(f'{OUTPUT_DIR}/*.json')):
    s = json.load(open(path))
    print(f"{s['mode']:<9} {s['model_name']:<26} "
          f"acc {s['accuracy_mean']:.3f}+/-{s['accuracy_std']:.3f}  "
          f"f1 {s['macro_f1_mean']:.3f}+/-{s['macro_f1_std']:.3f}")

In [ ]:
from IPython.display import Image, display
for path in sorted(glob.glob(f'{OUTPUT_DIR}/*_confusion.png')):
    display(Image(path))